In [39]:
import sys

print("Python do notebook:")
print(sys.executable)

from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm

print("\nReportLab OK")

Python do notebook:
C:\Users\beelt\Documents\collections_case_candidate\.venv\Scripts\python.exe

ReportLab OK


In [40]:
from pathlib import Path
import json
import io
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    Table,
    TableStyle,
    PageBreak,
    Image,
)

warnings.filterwarnings("ignore")

1. Configuração¶
Por padrão, o gerador procura os notebooks na mesma pasta deste arquivo ou na pasta anterior.
Ajuste apenas os caminhos se a estrutura do repositório mudar.

In [41]:
ROOT = Path.cwd()

CANDIDATES = [
    ROOT / "05_bivariate_eda_collections_macro_analysis.ipynb",
    ROOT.parent / "05_bivariate_eda_collections_macro_analysis.ipynb",
    ROOT / "notebooks" / "05_bivariate_eda_collections_macro_analysis.ipynb",
]

BIVARIATE_NOTEBOOK = next((p for p in CANDIDATES if p.exists()), None)

if BIVARIATE_NOTEBOOK is None:
    raise FileNotFoundError(
        "Não encontrei 05_bivariate_eda_collections_macro_analysis.ipynb. "
        "Ajuste BIVARIATE_NOTEBOOK nesta célula."
    )

REPORT_DIR = ROOT / "reports"
CHART_DIR = REPORT_DIR / "collections_report_charts"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
CHART_DIR.mkdir(parents=True, exist_ok=True)

PDF_PATH = REPORT_DIR / "collections_macro_bivariate_analysis_report.pdf"

print("Notebook fonte :", BIVARIATE_NOTEBOOK.resolve())
print("PDF de saída   :", PDF_PATH.resolve())

Notebook fonte : C:\Users\beelt\Documents\collections_case_candidate\notebooks\05_bivariate_eda_collections_macro_analysis.ipynb
PDF de saída   : C:\Users\beelt\Documents\collections_case_candidate\notebooks\reports\collections_macro_bivariate_analysis_report.pdf


In [42]:
pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_DIR = Path("../data")
QUEUE_PATH = DATA_DIR / "raw" / "collections_queue_sep2026.csv"
WA_PATH = DATA_DIR / "raw" / "whatsapp_collections_history.csv"

if not QUEUE_PATH.exists():
    QUEUE_PATH = Path("/mnt/data/collections_queue_sep2026(1).csv")
if not WA_PATH.exists():
    WA_PATH = Path("/mnt/data/whatsapp_collections_history(2).csv")

queue = pd.read_csv(QUEUE_PATH)
wa = pd.read_csv(WA_PATH)

wa["sent_at"] = pd.to_datetime(wa["sent_at"], errors="coerce")
queue["in_collections_since"] = pd.to_datetime(queue["in_collections_since"], errors="coerce")

print("Queue:", queue.shape)
print("WhatsApp:", wa.shape)

Queue: (10658, 10)
WhatsApp: (75406, 17)


In [43]:
##_______________________________________________________________________________________________________________________

In [44]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# ============================================================
# LOAD
# ============================================================


wa = wa

wa["sent_at"] = pd.to_datetime(wa["sent_at"])

wa = (
    wa
    .sort_values(["customer_id", "sent_at", "message_id"])
    .reset_index(drop=True)
)

print("=" * 100)
print("RAW WHATSAPP HISTORY")
print("=" * 100)

print(f"Rows       : {len(wa):,}")
print(f"Customers  : {wa['customer_id'].nunique():,}")
print(f"Date range : {wa['sent_at'].min()} → {wa['sent_at'].max()}")

print("\nCOLUMNS")
print(wa.dtypes)

print("\nMISSINGNESS")
print(
    wa.isna()
      .mean()
      .mul(100)
      .sort_values(ascending=False)
      .round(2)
)

RAW WHATSAPP HISTORY
Rows       : 75,406
Customers  : 11,724
Date range : 2026-06-01 09:18:00 → 2026-08-31 20:59:00

COLUMNS
message_id                           object
customer_id                          object
sent_at                      datetime64[ns]
template                             object
n_msgs_last_14d                       int64
days_past_due                         int64
outstanding_balance_brl             float64
monthly_salary_brl                  float64
payday_day_of_month                   int64
n_prior_transactions                  int64
account_age_months                    int64
days_since_last_app_login             int64
state_uf                             object
delivery_status                      object
interaction                          object
paid_within_72h                       int64
amount_paid_brl                     float64
dtype: object

MISSINGNESS
message_id                  0.00
customer_id                 0.00
sent_at                     0.00
t

In [45]:
# ============================================================
# 02 — RECONSTRUCT COLLECTION ENTRY DATE
# ============================================================

wa["sent_date"] = wa["sent_at"].dt.normalize()

wa["entry_date_reconstructed"] = (
    wa["sent_date"]
    - pd.to_timedelta(wa["days_past_due"] - 1, unit="D")
)

entry_consistency = (
    wa.groupby("customer_id")
      .agg(
          n_entry_dates=("entry_date_reconstructed", "nunique"),
          min_entry_date=("entry_date_reconstructed", "min"),
          max_entry_date=("entry_date_reconstructed", "max"),
          first_sent_at=("sent_at", "min"),
          last_sent_at=("sent_at", "max"),
          n_messages=("message_id", "count")
      )
      .reset_index()
)

print("=" * 100)
print("ENTRY DATE CONSISTENCY")
print("=" * 100)

print(
    entry_consistency["n_entry_dates"]
    .value_counts()
    .sort_index()
)

print("\nCustomers with inconsistent reconstructed entry date:")
print(
    (entry_consistency["n_entry_dates"] > 1).sum()
)

print("\nPercentage consistent:")
print(
    f"{(entry_consistency['n_entry_dates'].eq(1).mean() * 100):.2f}%"
)

ENTRY DATE CONSISTENCY
n_entry_dates
1    11724
Name: count, dtype: int64

Customers with inconsistent reconstructed entry date:
0

Percentage consistent:
100.00%


In [46]:
# ============================================================
# 03 — CUSTOMER ENTRY SNAPSHOT
# ============================================================

wa = wa.sort_values(
    ["customer_id", "sent_at", "message_id"]
).copy()

first_obs = (
    wa.groupby("customer_id", as_index=False)
      .first()
)

first_obs = first_obs[
    [
        "customer_id",
        "sent_at",
        "entry_date_reconstructed",
        "days_past_due",
        "outstanding_balance_brl",
        "monthly_salary_brl",
        "payday_day_of_month",
        "n_prior_transactions",
        "account_age_months",
        "days_since_last_app_login",
        "state_uf"
    ]
].copy()

first_obs = first_obs.rename(
    columns={
        "sent_at": "first_observed_at",
        "days_past_due": "entry_observed_dpd",
        "outstanding_balance_brl": "entry_balance_brl",
        "monthly_salary_brl": "entry_salary_brl",
        "payday_day_of_month": "entry_payday_day",
        "n_prior_transactions": "entry_n_prior_transactions",
        "account_age_months": "entry_account_age_months",
        "days_since_last_app_login": "entry_days_since_login",
        "state_uf": "entry_state_uf"
    }
)

first_obs["vintage_month"] = (
    first_obs["entry_date_reconstructed"]
    .dt.to_period("M")
    .astype(str)
)

print("=" * 100)
print("CUSTOMER ENTRY SNAPSHOT")
print("=" * 100)

print(f"Rows      : {len(first_obs):,}")
print(f"Customers : {first_obs['customer_id'].nunique():,}")

print("\nVintage distribution:")
print(
    first_obs["vintage_month"]
    .value_counts()
    .sort_index()
)

print("\nFirst observed DPD:")
print(
    first_obs["entry_observed_dpd"]
    .describe(
        percentiles=[.01, .05, .10, .25, .50, .75, .90, .95, .99]
    )
)

CUSTOMER ENTRY SNAPSHOT
Rows      : 11,724
Customers : 11,724

Vintage distribution:
vintage_month
2026-06    3966
2026-07    4100
2026-08    3658
Name: count, dtype: int64

First observed DPD:
count   11,724.00
mean         3.06
std          2.48
min          1.00
1%           1.00
5%           1.00
10%          1.00
25%          1.00
50%          2.00
75%          4.00
90%          6.00
95%          8.00
99%         12.00
max         32.00
Name: entry_observed_dpd, dtype: float64


In [47]:
# ============================================================
# 04 — PAYMENT / TARGET AUDIT
# ============================================================

print("=" * 100)
print("PAYMENT TARGET AUDIT")
print("=" * 100)

print("\npaid_within_72h:")
print(
    wa["paid_within_72h"]
    .value_counts(dropna=False)
)

print("\nAmount paid summary:")
print(
    wa["amount_paid_brl"]
    .describe(
        percentiles=[.50, .75, .90, .95, .99]
    )
)

print("\nCross-tab: paid_within_72h × amount_paid_brl > 0")

print(
    pd.crosstab(
        wa["paid_within_72h"],
        wa["amount_paid_brl"].gt(0),
        margins=True
    )
)

# ------------------------------------------------------------
# Customer-level naive aggregation
# ------------------------------------------------------------

payment_customer_audit = (
    wa.groupby("customer_id", as_index=False)
      .agg(
          messages=("message_id", "count"),
          payment_events=("paid_within_72h", "sum"),
          total_amount_paid_naive=("amount_paid_brl", "sum"),
          max_amount_paid=("amount_paid_brl", "max")
      )
)

payment_customer_audit["any_payment"] = (
    payment_customer_audit["total_amount_paid_naive"] > 0
).astype(int)

print("\nCUSTOMER LEVEL")
print(
    payment_customer_audit[
        [
            "messages",
            "payment_events",
            "total_amount_paid_naive",
            "any_payment"
        ]
    ].describe(
        percentiles=[.50, .75, .90, .95, .99]
    )
)

print("\nCustomers with any observed payment:")
print(
    payment_customer_audit["any_payment"]
    .value_counts()
)

print("\nNaive total amount:")
print(
    f"R$ {payment_customer_audit['total_amount_paid_naive'].sum():,.2f}"
)

PAYMENT TARGET AUDIT

paid_within_72h:
paid_within_72h
0    69780
1     5626
Name: count, dtype: int64

Amount paid summary:
count   75,406.00
mean        45.88
std        204.62
min          0.00
50%          0.00
75%          0.00
90%          0.00
95%        319.51
99%      1,152.02
max      2,000.00
Name: amount_paid_brl, dtype: float64

Cross-tab: paid_within_72h × amount_paid_brl > 0
amount_paid_brl  False  True    All
paid_within_72h                    
0                69780     0  69780
1                    0  5626   5626
All              69780  5626  75406

CUSTOMER LEVEL
       messages  payment_events  total_amount_paid_naive  any_payment
count 11,724.00       11,724.00                11,724.00    11,724.00
mean       6.43            0.48                   295.06         0.42
std        3.75            0.62                   462.95         0.49
min        1.00            0.00                     0.00         0.00
50%        6.00            0.00                     0.00     

In [48]:
# ============================================================
# 05 — TARGET MATURITY AUDIT
# ============================================================

DATA_END = wa["sent_at"].max().normalize()

maturity = (
    wa.groupby("customer_id", as_index=False)
      .agg(
          entry_date=("entry_date_reconstructed", "first"),
          first_message_at=("sent_at", "min"),
          last_message_at=("sent_at", "max"),
          n_messages=("message_id", "count"),
          total_recovery_brl=("amount_paid_brl", "sum"),
          any_recovery=("paid_within_72h", "max")
      )
)

maturity["vintage_month"] = (
    maturity["entry_date"]
    .dt.to_period("M")
    .astype(str)
)

# How many calendar days exist between entry and end of dataset?
maturity["potential_followup_days"] = (
    DATA_END - maturity["entry_date"]
).dt.days

print("=" * 100)
print("TARGET MATURITY AUDIT")
print("=" * 100)

print(f"Dataset end: {DATA_END.date()}")

print("\nPotential observation horizon by vintage:")

print(
    maturity
    .groupby("vintage_month")["potential_followup_days"]
    .describe(
        percentiles=[.01, .05, .10, .25, .50, .75, .90, .95, .99]
    )
    .round(1)
)

print("\nRecovery by vintage:")

vintage_recovery = (
    maturity
    .groupby("vintage_month", as_index=False)
    .agg(
        customers=("customer_id", "nunique"),
        payers=("any_recovery", "sum"),
        total_recovery_brl=("total_recovery_brl", "sum"),
        avg_followup_days=("potential_followup_days", "mean"),
        median_followup_days=("potential_followup_days", "median")
    )
)

vintage_recovery["payer_rate_pct"] = (
    100
    * vintage_recovery["payers"]
    / vintage_recovery["customers"]
)

vintage_recovery["recovery_per_customer"] = (
    vintage_recovery["total_recovery_brl"]
    / vintage_recovery["customers"]
)

print(vintage_recovery.to_string(index=False))

TARGET MATURITY AUDIT
Dataset end: 2026-08-31

Potential observation horizon by vintage:
                 count  mean  std   min    1%    5%   10%   25%   50%   75%   90%   95%   99%   max
vintage_month                                                                                      
2026-06       3,966.00 76.40 8.60 62.00 62.00 63.00 65.00 69.00 76.00 84.00 88.00 90.00 91.00 91.00
2026-07       4,100.00 46.00 8.90 31.00 31.00 32.00 34.00 38.00 46.00 54.00 58.00 60.00 61.00 61.00
2026-08       3,658.00 15.90 8.40  0.00  1.00  3.00  4.00  9.00 16.00 23.00 28.00 29.00 30.00 30.00

Recovery by vintage:
vintage_month  customers  payers  total_recovery_brl  avg_followup_days  median_followup_days  payer_rate_pct  recovery_per_customer
      2026-06       3966    1948        1,422,950.51              76.40                 76.00           49.12                 358.79
      2026-07       4100    1829        1,286,263.19              46.03                 46.00           44.61              

In [49]:
# ============================================================
# 06 — RECOVERY TIMING BY DPD
# ============================================================

payments = wa.loc[
    wa["amount_paid_brl"].gt(0),
    [
        "customer_id",
        "sent_at",
        "entry_date_reconstructed",
        "days_past_due",
        "amount_paid_brl"
    ]
].copy()

print("=" * 100)
print("RECOVERY TIMING BY DPD")
print("=" * 100)

print(f"Payment events : {len(payments):,}")
print(f"Payers         : {payments['customer_id'].nunique():,}")
print(f"Total recovery : R$ {payments['amount_paid_brl'].sum():,.2f}")

print("\nDPD of message associated with payment:")
print(
    payments["days_past_due"]
    .describe(
        percentiles=[
            .10, .25, .50, .75, .80,
            .85, .90, .95, .97, .99
        ]
    )
)

# ============================================================
# CUMULATIVE PAYMENT EVENTS + RECOVERY
# ============================================================

thresholds = [1, 3, 5, 7, 10, 15, 20, 30, 45, 60]

total_recovery = payments["amount_paid_brl"].sum()
total_events = len(payments)
total_payers = payments["customer_id"].nunique()

rows = []

for dpd in thresholds:

    p = payments.loc[
        payments["days_past_due"] <= dpd
    ]

    rows.append({
        "through_dpd": dpd,
        "payment_events": len(p),
        "pct_payment_events": 100 * len(p) / total_events,
        "payers": p["customer_id"].nunique(),
        "pct_eventual_payers_reached":
            100 * p["customer_id"].nunique() / total_payers,
        "recovery_brl": p["amount_paid_brl"].sum(),
        "pct_total_recovery":
            100 * p["amount_paid_brl"].sum() / total_recovery
    })

recovery_timing = pd.DataFrame(rows)

print("\nCUMULATIVE RECOVERY BY DPD")

print(
    recovery_timing.to_string(
        index=False,
        formatters={
            "pct_payment_events": "{:.2f}%".format,
            "pct_eventual_payers_reached": "{:.2f}%".format,
            "recovery_brl": "R$ {:,.2f}".format,
            "pct_total_recovery": "{:.2f}%".format
        }
    )
)

RECOVERY TIMING BY DPD
Payment events : 5,626
Payers         : 4,893
Total recovery : R$ 3,459,305.30

DPD of message associated with payment:
count   5,626.00
mean       14.72
std        13.45
min         1.00
10%         2.00
25%         5.00
50%        10.00
75%        21.00
80%        25.00
85%        29.00
90%        36.00
95%        44.00
97%        50.00
99%        56.00
max        60.00
Name: days_past_due, dtype: float64

CUMULATIVE RECOVERY BY DPD
 through_dpd  payment_events pct_payment_events  payers pct_eventual_payers_reached    recovery_brl pct_total_recovery
           1             386              6.86%     386                       7.89%   R$ 256,870.70              7.43%
           3            1062             18.88%    1062                      21.70%   R$ 695,994.91             20.12%
           5            1635             29.06%    1632                      33.35% R$ 1,084,415.21             31.35%
           7            2150             38.22%    2122       

In [50]:
# ============================================================
# 07 — MATURITY AT CANDIDATE TARGET WINDOWS
# ============================================================

windows = [7, 15, 30, 45, 60]

rows = []

for h in windows:

    tmp = maturity.copy()

    tmp["mature"] = (
        tmp["potential_followup_days"] >= h
    )

    for vintage, g in tmp.groupby("vintage_month"):

        rows.append({
            "target_window_days": h,
            "vintage": vintage,
            "customers": len(g),
            "mature_customers": int(g["mature"].sum()),
            "immature_customers": int((~g["mature"]).sum()),
            "maturity_rate_pct": 100 * g["mature"].mean()
        })

maturity_table = pd.DataFrame(rows)

print("=" * 100)
print("TARGET MATURITY BY WINDOW")
print("=" * 100)

print(
    maturity_table.to_string(
        index=False,
        formatters={
            "maturity_rate_pct": "{:.2f}%".format
        }
    )
)

# Easier comparison
print("\n" + "=" * 100)
print("MATURE CUSTOMERS — PIVOT")
print("=" * 100)

print(
    maturity_table.pivot(
        index="target_window_days",
        columns="vintage",
        values="mature_customers"
    )
)

print("\n" + "=" * 100)
print("MATURITY RATE — PIVOT")
print("=" * 100)

print(
    maturity_table.pivot(
        index="target_window_days",
        columns="vintage",
        values="maturity_rate_pct"
    ).round(2)
)

TARGET MATURITY BY WINDOW
 target_window_days vintage  customers  mature_customers  immature_customers maturity_rate_pct
                  7 2026-06       3966              3966                   0           100.00%
                  7 2026-07       4100              4100                   0           100.00%
                  7 2026-08       3658              3034                 624            82.94%
                 15 2026-06       3966              3966                   0           100.00%
                 15 2026-07       4100              4100                   0           100.00%
                 15 2026-08       3658              2033                1625            55.58%
                 30 2026-06       3966              3966                   0           100.00%
                 30 2026-07       4100              4100                   0           100.00%
                 30 2026-08       3658               129                3529             3.53%
                 45 2026

In [51]:
# ============================================================
# 08 — OFFICIAL TARGET: RECOVERY THROUGH DPD30
# ============================================================

H = 30

target_30 = (
    wa.loc[
        wa["days_past_due"].between(1, H)
    ]
    .groupby("customer_id", as_index=False)
    .agg(
        recovery_30d_brl=("amount_paid_brl", "sum"),
        payment_events_30d=("paid_within_72h", "sum")
    )
)

# Customers with no observed payment/event still need to exist
target_30 = (
    wa[["customer_id"]]
    .drop_duplicates()
    .merge(
        target_30,
        on="customer_id",
        how="left",
        validate="1:1"
    )
)

target_30[
    ["recovery_30d_brl", "payment_events_30d"]
] = target_30[
    ["recovery_30d_brl", "payment_events_30d"]
].fillna(0)

target_30["any_recovery_30d"] = (
    target_30["recovery_30d_brl"] > 0
).astype(int)


print("=" * 100)
print("OFFICIAL DPD30 TARGET")
print("=" * 100)

print(f"Customers        : {len(target_30):,}")
print(f"Payers ≤ DPD30   : {target_30['any_recovery_30d'].sum():,}")
print(
    f"Payment rate     : "
    f"{100 * target_30['any_recovery_30d'].mean():.2f}%"
)
print(
    f"Recovery ≤ DPD30 : "
    f"R$ {target_30['recovery_30d_brl'].sum():,.2f}"
)

OFFICIAL DPD30 TARGET
Customers        : 11,724
Payers ≤ DPD30   : 4,355
Payment rate     : 37.15%
Recovery ≤ DPD30 : R$ 3,028,175.92


In [52]:
# ============================================================
# 09 — MODELING BASE
# 1 ROW = CUSTOMER AT ENTRY
# ============================================================

model_df = (
    first_obs
    .merge(
        target_30,
        on="customer_id",
        how="left",
        validate="1:1"
    )
)

# maturity
DATA_END = wa["sent_at"].max().normalize()

model_df["available_followup_days"] = (
    DATA_END
    - model_df["entry_date_reconstructed"]
).dt.days

model_df["mature_30d"] = (
    model_df["available_followup_days"] >= 30
).astype(int)


print("=" * 100)
print("MODELING BASE")
print("=" * 100)

print(f"Customers : {len(model_df):,}")

print("\nBy vintage:")

audit = (
    model_df
    .groupby("vintage_month")
    .agg(
        customers=("customer_id", "count"),
        mature_30d=("mature_30d", "sum"),
        payers_30d=("any_recovery_30d", "sum"),
        recovery_30d_brl=("recovery_30d_brl", "sum")
    )
)

audit["raw_payment_rate_pct"] = (
    100
    * audit["payers_30d"]
    / audit["customers"]
)

print(audit)

MODELING BASE
Customers : 11,724

By vintage:
               customers  mature_30d  payers_30d  recovery_30d_brl  raw_payment_rate_pct
vintage_month                                                                           
2026-06             3966        3966        1618      1,162,479.93                 40.80
2026-07             4100        4100        1621      1,115,604.39                 39.54
2026-08             3658         129        1116        750,091.60                 30.51


In [53]:
# ============================================================
# 12 — OFFICIAL DPD30 MODELING DATASET
# ============================================================

H = 30

# ------------------------------------------------------------
# TARGET
# ------------------------------------------------------------

target_30 = (
    wa.loc[
        wa["days_past_due"].between(1, H)
    ]
    .groupby("customer_id", as_index=False)
    .agg(
        recovery_30d_brl=("amount_paid_brl", "sum"),
        payment_events_30d=("paid_within_72h", "sum")
    )
)

target_30 = (
    wa[["customer_id"]]
    .drop_duplicates()
    .merge(
        target_30,
        on="customer_id",
        how="left",
        validate="1:1"
    )
)

target_30[
    ["recovery_30d_brl", "payment_events_30d"]
] = (
    target_30[
        ["recovery_30d_brl", "payment_events_30d"]
    ]
    .fillna(0)
)

target_30["any_recovery_30d"] = (
    target_30["recovery_30d_brl"] > 0
).astype(int)


# ------------------------------------------------------------
# MERGE ENTRY SNAPSHOT + TARGET
# ------------------------------------------------------------

model_df = (
    first_obs
    .merge(
        target_30,
        on="customer_id",
        how="left",
        validate="1:1"
    )
)


# ------------------------------------------------------------
# ONLY MATURE VINTAGES
# ------------------------------------------------------------

model_df = model_df[
    model_df["vintage_month"].isin(
        ["2026-06", "2026-07"]
    )
].copy()


# ------------------------------------------------------------
# SPLIT
# ------------------------------------------------------------

train = model_df[
    model_df["vintage_month"] == "2026-06"
].copy()

valid = model_df[
    model_df["vintage_month"] == "2026-07"
].copy()


print("=" * 100)
print("OFFICIAL MODELING POPULATION")
print("=" * 100)

for name, df in [
    ("TRAIN — JUN", train),
    ("VALIDATION — JUL", valid)
]:

    print(f"\n{name}")
    print("-" * 60)

    print(f"Customers       : {len(df):,}")
    print(
        f"Payers ≤ DPD30  : "
        f"{df['any_recovery_30d'].sum():,}"
    )
    print(
        f"Payment rate    : "
        f"{100 * df['any_recovery_30d'].mean():.2f}%"
    )
    print(
        f"Recovery ≤ DPD30: "
        f"R$ {df['recovery_30d_brl'].sum():,.2f}"
    )
    print(
        f"Recovery/customer: "
        f"R$ {df['recovery_30d_brl'].mean():,.2f}"
    )

OFFICIAL MODELING POPULATION

TRAIN — JUN
------------------------------------------------------------
Customers       : 3,966
Payers ≤ DPD30  : 1,618
Payment rate    : 40.80%
Recovery ≤ DPD30: R$ 1,162,479.93
Recovery/customer: R$ 293.11

VALIDATION — JUL
------------------------------------------------------------
Customers       : 4,100
Payers ≤ DPD30  : 1,621
Payment rate    : 39.54%
Recovery ≤ DPD30: R$ 1,115,604.39
Recovery/customer: R$ 272.10


In [54]:
# ============================================================
# 13 — ENTRY FEATURE AUDIT
# ============================================================

candidate_features = [
    "entry_observed_dpd",
    "entry_balance_brl",
    "entry_salary_brl",
    "entry_payday_day",
    "entry_n_prior_transactions",
    "entry_account_age_months",
    "entry_days_since_login",
    "entry_state_uf"
]

print("=" * 100)
print("ENTRY FEATURE AUDIT")
print("=" * 100)

print("\nDtypes:")
print(
    model_df[candidate_features].dtypes
)

print("\nMissingness:")
print(
    model_df[candidate_features]
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .round(2)
)

print("\nNumeric summary:")

numeric_features = [
    "entry_observed_dpd",
    "entry_balance_brl",
    "entry_salary_brl",
    "entry_payday_day",
    "entry_n_prior_transactions",
    "entry_account_age_months",
    "entry_days_since_login"
]

print(
    model_df[numeric_features]
    .describe(
        percentiles=[
            .01, .05, .10, .25,
            .50, .75, .90, .95, .99
        ]
    )
    .T
)

print("\nState distribution:")
print(
    model_df["entry_state_uf"]
    .value_counts(dropna=False)
)

ENTRY FEATURE AUDIT

Dtypes:
entry_observed_dpd              int64
entry_balance_brl             float64
entry_salary_brl              float64
entry_payday_day                int64
entry_n_prior_transactions      int64
entry_account_age_months        int64
entry_days_since_login          int64
entry_state_uf                 object
dtype: object

Missingness:
entry_observed_dpd           0.00
entry_balance_brl            0.00
entry_salary_brl             0.00
entry_payday_day             0.00
entry_n_prior_transactions   0.00
entry_account_age_months     0.00
entry_days_since_login       0.00
entry_state_uf               0.00
dtype: float64

Numeric summary:
                              count     mean      std      min       1%       5%      10%      25%      50%      75%      90%      95%      99%       max
entry_observed_dpd         8,066.00     3.14     2.58     1.00     1.00     1.00     1.00     1.00     2.00     4.00     6.00     8.00    12.00     32.00
entry_balance_brl         

In [55]:
# ============================================================
# 14 — ENTRY DERIVED FEATURES
# ============================================================

model_df["balance_to_salary_ratio"] = np.where(
    model_df["entry_salary_brl"] > 0,
    model_df["entry_balance_brl"]
    / model_df["entry_salary_brl"],
    np.nan
)

train = model_df[
    model_df["vintage_month"] == "2026-06"
].copy()

valid = model_df[
    model_df["vintage_month"] == "2026-07"
].copy()

print("=" * 100)
print("BALANCE / SALARY")
print("=" * 100)

print(
    model_df["balance_to_salary_ratio"]
    .describe(
        percentiles=[
            .01, .05, .10, .25,
            .50, .75, .90, .95, .99
        ]
    )
)

BALANCE / SALARY
count   8,066.00
mean        0.33
std         0.14
min         0.06
1%          0.09
5%          0.12
10%         0.15
25%         0.21
50%         0.32
75%         0.43
90%         0.51
95%         0.55
99%         0.63
max         0.83
Name: balance_to_salary_ratio, dtype: float64


In [56]:
# ============================================================
# 15 — ENTRY-POINT DERIVED FEATURES
# ============================================================

model_df["entry_day_of_month"] = (
    model_df["entry_date_reconstructed"].dt.day
)

model_df["entry_day_of_week"] = (
    model_df["entry_date_reconstructed"].dt.dayofweek
)

model_df["entry_is_weekend"] = (
    model_df["entry_day_of_week"] >= 5
).astype(int)


# ------------------------------------------------------------
# BALANCE / SALARY
# ------------------------------------------------------------

model_df["balance_to_salary_ratio"] = (
    model_df["entry_balance_brl"]
    / model_df["entry_salary_brl"]
)


# ------------------------------------------------------------
# DAYS TO NEXT PAYDAY
# Circular distance within an approximate 30-day salary cycle
# ------------------------------------------------------------

model_df["days_to_next_payday"] = (
    model_df["entry_payday_day"]
    - model_df["entry_day_of_month"]
) % 30


# ------------------------------------------------------------
# PAYDAY PROXIMITY FLAGS
# ------------------------------------------------------------

model_df["payday_within_3d"] = (
    model_df["days_to_next_payday"] <= 3
).astype(int)

model_df["payday_within_7d"] = (
    model_df["days_to_next_payday"] <= 7
).astype(int)


print("=" * 100)
print("DERIVED ENTRY FEATURES")
print("=" * 100)

cols = [
    "balance_to_salary_ratio",
    "days_to_next_payday",
    "entry_day_of_month",
    "entry_day_of_week",
    "entry_is_weekend",
    "payday_within_3d",
    "payday_within_7d"
]

print(
    model_df[cols]
    .describe(
        percentiles=[
            .01, .05, .10, .25,
            .50, .75, .90, .95, .99
        ]
    )
    .T
)

DERIVED ENTRY FEATURES
                           count  mean  std  min   1%   5%  10%  25%   50%   75%   90%   95%   99%   max
balance_to_salary_ratio 8,066.00  0.33 0.14 0.06 0.09 0.12 0.15 0.21  0.32  0.43  0.51  0.55  0.63  0.83
days_to_next_payday     8,066.00 14.55 8.62 0.00 0.00 1.00 3.00 7.00 15.00 22.00 26.00 28.00 29.00 29.00
entry_day_of_month      8,066.00 15.79 8.76 1.00 1.00 2.00 4.00 8.00 16.00 23.00 28.00 29.00 31.00 31.00
entry_day_of_week       8,066.00  2.93 1.98 0.00 0.00 0.00 0.00 1.00  3.00  5.00  6.00  6.00  6.00  6.00
entry_is_weekend        8,066.00  0.27 0.44 0.00 0.00 0.00 0.00 0.00  0.00  1.00  1.00  1.00  1.00  1.00
payday_within_3d        8,066.00  0.13 0.33 0.00 0.00 0.00 0.00 0.00  0.00  0.00  1.00  1.00  1.00  1.00
payday_within_7d        8,066.00  0.26 0.44 0.00 0.00 0.00 0.00 0.00  0.00  1.00  1.00  1.00  1.00  1.00


In [57]:
# ============================================================
# 16 — FINAL TEMPORAL SPLIT
# ============================================================

train = model_df[
    model_df["vintage_month"] == "2026-06"
].copy()

valid = model_df[
    model_df["vintage_month"] == "2026-07"
].copy()

print("=" * 100)
print("TEMPORAL SPLIT")
print("=" * 100)

for name, df in [
    ("TRAIN — JUN", train),
    ("VALIDATION — JUL", valid)
]:
    
    print(f"\n{name}")
    print(f"Customers       : {len(df):,}")
    print(
        f"Payers DPD30    : "
        f"{df['any_recovery_30d'].sum():,}"
    )
    print(
        f"Payment rate    : "
        f"{100 * df['any_recovery_30d'].mean():.2f}%"
    )
    print(
        f"Recovery DPD30  : "
        f"R$ {df['recovery_30d_brl'].sum():,.2f}"
    )
    print(
        f"Recovery/customer: "
        f"R$ {df['recovery_30d_brl'].mean():,.2f}"
    )

TEMPORAL SPLIT

TRAIN — JUN
Customers       : 3,966
Payers DPD30    : 1,618
Payment rate    : 40.80%
Recovery DPD30  : R$ 1,162,479.93
Recovery/customer: R$ 293.11

VALIDATION — JUL
Customers       : 4,100
Payers DPD30    : 1,621
Payment rate    : 39.54%
Recovery DPD30  : R$ 1,115,604.39
Recovery/customer: R$ 272.10


In [58]:
# ============================================================
# 17 — FEATURE SET
# ============================================================

numeric_features = [
    "entry_observed_dpd",
    "entry_balance_brl",
    "entry_salary_brl",
    "entry_n_prior_transactions",
    "entry_account_age_months",
    "entry_days_since_login",
    "balance_to_salary_ratio",
    "days_to_next_payday",
    "entry_day_of_month",
    "entry_day_of_week"
]

categorical_features = [
    "entry_state_uf"
]

TARGET = "any_recovery_30d"
RECOVERY = "recovery_30d_brl"

In [59]:
# ============================================================
# 18 — LOGISTIC REGRESSION L2
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss
)

# ------------------------------------------------------------
# PREPROCESSING
# ------------------------------------------------------------

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numeric_features
        ),
        (
            "cat",
            categorical_transformer,
            categorical_features
        )
    ]
)


# ------------------------------------------------------------
# MODEL
# ------------------------------------------------------------

logit_l2 = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            LogisticRegression(
                penalty="l2",
                C=1.0,
                solver="liblinear",
                max_iter=2000,
                random_state=42
            )
        )
    ]
)


X_train = train[numeric_features + categorical_features]
y_train = train[TARGET]

X_valid = valid[numeric_features + categorical_features]
y_valid = valid[TARGET]


logit_l2.fit(
    X_train,
    y_train
)


train["score_logit_l2"] = (
    logit_l2.predict_proba(X_train)[:, 1]
)

valid["score_logit_l2"] = (
    logit_l2.predict_proba(X_valid)[:, 1]
)

In [60]:
# ============================================================
# 19 — CLASSIFICATION PERFORMANCE
# ============================================================

from scipy.stats import ks_2samp


def evaluate_classifier(df, target, score):

    y = df[target]
    p = df[score]

    auc = roc_auc_score(y, p)

    ap = average_precision_score(y, p)

    ll = log_loss(y, p)

    positive_scores = df.loc[
        df[target].eq(1),
        score
    ]

    negative_scores = df.loc[
        df[target].eq(0),
        score
    ]

    ks = ks_2samp(
        positive_scores,
        negative_scores
    ).statistic

    return {
        "customers": len(df),
        "payer_rate": y.mean(),
        "roc_auc": auc,
        "pr_auc": ap,
        "ks": ks,
        "log_loss": ll
    }


print("=" * 100)
print("LOGISTIC L2 — PERFORMANCE")
print("=" * 100)

print("\nTRAIN")
print(
    evaluate_classifier(
        train,
        TARGET,
        "score_logit_l2"
    )
)

print("\nVALIDATION")
print(
    evaluate_classifier(
        valid,
        TARGET,
        "score_logit_l2"
    )
)

LOGISTIC L2 — PERFORMANCE

TRAIN
{'customers': 3966, 'payer_rate': np.float64(0.4079677256681795), 'roc_auc': 0.6024349681921652, 'pr_auc': 0.49649467709120404, 'ks': np.float64(0.14502203700701014), 'log_loss': 0.6592960168947709}

VALIDATION
{'customers': 4100, 'payer_rate': np.float64(0.3953658536585366), 'roc_auc': 0.5732351630314009, 'pr_auc': 0.4447366656079865, 'ks': np.float64(0.11452424922090781), 'log_loss': 0.664492089619565}


In [61]:
# ============================================================
# 20 — DEPRIORITIZATION CURVE
# ============================================================

def deprioritization_curve(
    df,
    score_col,
    recovery_col,
    cuts=None
):

    if cuts is None:
        cuts = [
            0.05, 0.10, 0.15, 0.20,
            0.25, 0.30, 0.35, 0.40,
            0.45, 0.50
        ]

    d = (
        df
        .sort_values(
            score_col,
            ascending=True
        )
        .reset_index(drop=True)
        .copy()
    )

    total_customers = len(d)
    total_recovery = d[recovery_col].sum()

    rows = []

    for cut in cuts:

        n_remove = int(
            np.floor(
                total_customers * cut
            )
        )

        removed = d.iloc[:n_remove]
        retained = d.iloc[n_remove:]

        lost_recovery = (
            removed[recovery_col].sum()
        )

        retained_recovery = (
            retained[recovery_col].sum()
        )

        rows.append({
            "removed_pct": 100 * cut,
            "customers_removed": n_remove,

            "payers_removed":
                removed[TARGET].sum(),

            "payer_rate_removed_pct":
                100 * removed[TARGET].mean(),

            "recovery_lost_brl":
                lost_recovery,

            "recovery_lost_pct":
                100 * lost_recovery
                / total_recovery,

            "recovery_preserved_pct":
                100 * retained_recovery
                / total_recovery
        })

    return pd.DataFrame(rows)


deprio_valid = deprioritization_curve(
    valid,
    score_col="score_logit_l2",
    recovery_col=RECOVERY
)

print("=" * 100)
print("VALIDATION — DEPRIORITIZATION CURVE")
print("=" * 100)

print(
    deprio_valid.to_string(
        index=False,
        formatters={
            "removed_pct":
                "{:.0f}%".format,

            "payer_rate_removed_pct":
                "{:.2f}%".format,

            "recovery_lost_brl":
                "R$ {:,.2f}".format,

            "recovery_lost_pct":
                "{:.2f}%".format,

            "recovery_preserved_pct":
                "{:.2f}%".format
        }
    )
)

VALIDATION — DEPRIORITIZATION CURVE
removed_pct  customers_removed  payers_removed payer_rate_removed_pct recovery_lost_brl recovery_lost_pct recovery_preserved_pct
         5%                205              44                 21.46%      R$ 35,625.88             3.19%                 96.81%
        10%                410             103                 25.12%      R$ 80,773.23             7.24%                 92.76%
        15%                615             176                 28.62%     R$ 144,952.74            12.99%                 87.01%
        20%                820             236                 28.78%     R$ 202,499.17            18.15%                 81.85%
        25%               1025             319                 31.12%     R$ 271,930.06            24.38%                 75.62%
        30%               1230             400                 32.52%     R$ 342,244.62            30.68%                 69.32%
        35%               1435             473               

In [62]:
# ============================================================
# 21 — ECONOMIC DEPRIORITIZATION BENCHMARK
# ============================================================

def economic_curve(
    df,
    ranking_col,
    ascending=True,
    cuts=None
):
    """
    Simulates deprioritizing customers according to a ranking.

    ascending=True:
        lowest values are removed first

    ascending=False:
        highest values are removed first
    """

    if cuts is None:
        cuts = [
            0.05, 0.10, 0.15, 0.20,
            0.25, 0.30, 0.35, 0.40,
            0.45, 0.50
        ]

    d = (
        df
        .sort_values(
            ranking_col,
            ascending=ascending
        )
        .reset_index(drop=True)
        .copy()
    )

    total_n = len(d)
    total_recovery = d["recovery_30d_brl"].sum()

    rows = []

    for cut in cuts:

        n_remove = int(
            np.floor(total_n * cut)
        )

        removed = d.iloc[:n_remove]

        recovery_lost = (
            removed["recovery_30d_brl"].sum()
        )

        rows.append({
            "removed_pct": int(cut * 100),

            "customers_removed": n_remove,

            "payer_rate_removed_pct":
                100
                * removed["any_recovery_30d"].mean(),

            "recovery_lost_brl":
                recovery_lost,

            "recovery_lost_pct":
                100
                * recovery_lost
                / total_recovery,

            "recovery_preserved_pct":
                100
                * (
                    1
                    - recovery_lost
                    / total_recovery
                )
        })

    return pd.DataFrame(rows)


# ============================================================
# STRATEGIES
# ============================================================

strategies = {

    "Logistic P(pay)": (
        "score_logit_l2",
        True
    ),

    "Lowest balance": (
        "entry_balance_brl",
        True
    ),

    "Highest balance/salary": (
        "balance_to_salary_ratio",
        False
    ),

    "Longest since login": (
        "entry_days_since_login",
        False
    ),

    "Lowest salary": (
        "entry_salary_brl",
        True
    )
}


# ============================================================
# RUN ALL STRATEGIES
# ============================================================

benchmark_rows = []

for strategy, (col, ascending) in strategies.items():

    curve = economic_curve(
        valid,
        ranking_col=col,
        ascending=ascending
    )

    curve["strategy"] = strategy

    benchmark_rows.append(curve)


benchmark = pd.concat(
    benchmark_rows,
    ignore_index=True
)


print("Benchmark created successfully.")
print(f"Rows: {len(benchmark):,}")

display(benchmark.head())

Benchmark created successfully.
Rows: 50


,removed_pct,customers_removed,payer_rate_removed_pct,recovery_lost_brl,recovery_lost_pct,recovery_preserved_pct,strategy
0,5,205,21.46,"35,625.88",3.19,96.81,Logistic P(pay)
1,10,410,25.12,"80,773.23",7.24,92.76,Logistic P(pay)
2,15,615,28.62,"144,952.74",12.99,87.01,Logistic P(pay)
3,20,820,28.78,"202,499.17",18.15,81.85,Logistic P(pay)
4,25,1025,31.12,"271,930.06",24.38,75.62,Logistic P(pay)


In [63]:
# ============================================================
# RECOVERY LOST
# ============================================================

benchmark_pivot = (
    benchmark
    .pivot(
        index="removed_pct",
        columns="strategy",
        values="recovery_lost_pct"
    )
)

print("=" * 100)
print("RECOVERY LOST % — STRATEGY COMPARISON")
print("=" * 100)

print(
    benchmark_pivot.round(2)
)

RECOVERY LOST % — STRATEGY COMPARISON
strategy     Highest balance/salary  Logistic P(pay)  Longest since login  Lowest balance  Lowest salary
removed_pct                                                                                             
5                              5.95             3.19                 4.01            2.26           2.77
10                            11.64             7.24                 8.70            4.21           5.40
15                            19.26            12.99                12.59            6.40           8.21
20                            23.50            18.15                17.79            9.14          11.19
25                            30.31            24.38                23.11           11.83          15.12
30                            37.03            30.68                27.67           14.89          18.45
35                            43.43            35.71                32.02           18.83          21.90
40               

In [64]:
# ============================================================
# 23 — SIMPLE EXPECTED VALUE SCORE
# ============================================================

train["simple_ev_score"] = (
    train["score_logit_l2"]
    * train["entry_balance_brl"]
)

valid["simple_ev_score"] = (
    valid["score_logit_l2"]
    * valid["entry_balance_brl"]
)

ev_curve = economic_curve(
    valid,
    ranking_col="simple_ev_score",
    ascending=True
)

print("=" * 100)
print("P(PAY) × BALANCE — DEPRIORITIZATION")
print("=" * 100)

print(
    ev_curve.to_string(
        index=False,
        formatters={
            "payer_rate_removed_pct":
                "{:.2f}%".format,

            "recovery_lost_brl":
                "R$ {:,.2f}".format,

            "recovery_lost_pct":
                "{:.2f}%".format,

            "recovery_preserved_pct":
                "{:.2f}%".format
        }
    )
)

P(PAY) × BALANCE — DEPRIORITIZATION
 removed_pct  customers_removed payer_rate_removed_pct recovery_lost_brl recovery_lost_pct recovery_preserved_pct
           5                205                 38.05%      R$ 18,529.98             1.66%                 98.34%
          10                410                 43.90%      R$ 42,433.38             3.80%                 96.20%
          15                615                 45.53%      R$ 70,611.58             6.33%                 93.67%
          20                820                 45.12%     R$ 100,066.87             8.97%                 91.03%
          25               1025                 43.32%     R$ 129,797.67            11.63%                 88.37%
          30               1230                 42.11%     R$ 160,148.83            14.36%                 85.64%
          35               1435                 41.67%     R$ 198,173.49            17.76%                 82.24%
          40               1640                 41.8